In [24]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [25]:
from pathlib import Path

In [26]:
# Get the project directory
PROJECT_DIR = Path.cwd().parents[0]

In [27]:
# Sizes
IMG_SIZE = 64
BATCH_SIZE = 3

In [28]:
train_ds = keras.utils.image_dataset_from_directory(
    PROJECT_DIR / "data",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 27 files belonging to 3 classes.
Using 22 files for training.


In [29]:
# Validation dataset
val_ds = keras.utils.image_dataset_from_directory(
    PROJECT_DIR / "data",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 27 files belonging to 3 classes.
Using 5 files for validation.


In [30]:
class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)

Classes: ['applegothic', 'arial', 'helvetica']


In [31]:
# Cache

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [32]:
# Augmentation
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
])

In [33]:
normalization = layers.Rescaling(1./255)

In [34]:
# Customize a model from the base MobileNetV2 model
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)

/var/folders/vb/gdmzt8xs5bb2rp7k5ht40ync0000gn/T/ipykernel_13044/400901966.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(


In [35]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Summary of the model
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_2 (Sequential)       │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 2, 2, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,163 (8.93 MB)

 Trainable params: 82,179 (321.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [36]:
# Create a callback to stop training when the improvement metric has stopped improving
callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', # set the improvement metric to value loss
    patience=3, # the number of epochs to wait
    restore_best_weights=True   # roll back the model to its best performance state
)

In [37]:
history = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=[callback]
)

Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 372ms/step - accuracy: 0.2727 - loss: 2.1818 - val_accuracy: 0.4000 - val_loss: 2.2395
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.2727 - loss: 2.1719 - val_accuracy: 0.4000 - val_loss: 2.1504
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5000 - loss: 1.5800 - val_accuracy: 0.4000 - val_loss: 2.0516
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5000 - loss: 1.5275 - val_accuracy: 0.4000 - val_loss: 1.8861
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3182 - loss: 1.5975 - val_accuracy: 0.4000 - val_loss: 1.7991
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3182 - loss: 1.6288 - val_accuracy: 0.2000 - val_loss: 1.7792
Epoch 7/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.4091 - loss: 1.5436 - val_accuracy: 0.2000 - val_loss: 1.7739
Epoch 8/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3636 - loss: 1.4158 - val_accuracy: 0.2000 - val_loss: 1.7136

In [38]:
# Evaluate the model
test_loss, test_acc = model.evaluate(val_ds, verbose=2)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

2/2 - 0s - 16ms/step - accuracy: 0.2000 - loss: 1.4952

Test Accuracy: 20.00%


In [39]:
# Save the model and its parameters
# model.save('../models/font_recognition_MobileNetV2.keras')